# Multimodal RAG over the Alfa Romeo Giulia Owner's Manual

A retrieval-augmented generation (RAG) pipeline built on top of a 360-page automotive
owner's manual. The system answers technical questions in natural language, cites the
exact manual page it used, and returns the matching figure or page scan alongside the
answer.

**Pipeline overview**

| Stage | Tooling | Purpose |
|---|---|---|
| 1. Parse | PyMuPDF | Extract per-page text blocks and embedded figures |
| 2. Render | pdf2image + poppler | Render every page to PNG for visual citations |
| 3. Chunk | LangChain `RecursiveCharacterTextSplitter` | 600-char chunks, 100-char overlap |
| 4. Embed & store | `all-MiniLM-L6-v2` + ChromaDB | Persistent vector store on Google Drive |
| 5. Generate | Ollama (`qwen2.5:14b`, `llava:13b`) | Grounded answers + visual figure analysis |
| 6. Serve | FastAPI + ngrok | REST API with static image mounts |

**Environment:** Google Colab (T4 GPU), Google Drive for persistence, all models run
locally through Ollama, so no external LLM API key is required.

> **Configuration:** the ngrok tunnel is the only component that needs a credential.
> Set it as an environment variable (or a Colab secret) before running Section 8 —
> no token is hard-coded anywhere in this notebook.

---
## 1. Environment setup and project paths

Mounts Google Drive and defines a single source of truth for every path used later.
All artifacts (extracted images, page renders, vector store, backend code) live under
`BASE_DIR` so the project survives Colab runtime restarts.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive as the persistence layer for this project
drive.mount('/content/drive')

# Root of the project inside Drive
BASE_DIR = '/content/drive/MyDrive/RAG_Project'
DATA_DIR = os.path.join(BASE_DIR, 'data')
IMAGE_OUTPUT_DIR = os.path.join(DATA_DIR, 'extracted_images')  # figures embedded in the PDF
PAGES_DIR = os.path.join(DATA_DIR, 'manual_pages')             # full-page PNG renders
VECTOR_DB_DIR = os.path.join(BASE_DIR, 'vector_store')         # ChromaDB persistent store
APP_DIR = os.path.join(BASE_DIR, 'backend', 'app')             # FastAPI application folder

# Create the folder structure if this is a first run
for path in (IMAGE_OUTPUT_DIR, PAGES_DIR, VECTOR_DB_DIR, APP_DIR):
    os.makedirs(path, exist_ok=True)

# Source document
PDF_PATH = os.path.join(DATA_DIR, 'alfa_romeo_giulia_manual.pdf')

print("=== Paths configured ===")
print(f"Project base directory : {BASE_DIR}")
print(f"PDF source             : {PDF_PATH}")
print(f"Extracted figures      : {IMAGE_OUTPUT_DIR}")
print(f"Page renders           : {PAGES_DIR}")
print(f"Vector store           : {VECTOR_DB_DIR}")

In [ ]:
# Core dependencies for parsing, chunking, embedding and vector storage
!pip install -q PyMuPDF chromadb sentence-transformers langchain-text-splitters

---
## 2. PDF parsing: text blocks and embedded figures

For each page the parser does two things:

1. **Figure extraction** — every embedded raster image is written to disk as
   `page_{n}_fig_{i}.{ext}`. The filename encodes the page number, which is what lets the
   API later match a retrieved chunk back to its illustration without a second index.
2. **Layout-aware text extraction** — blocks are sorted by `(x0, y0)` rather than reading
   order, because the manual uses a two-column layout and naive extraction interleaves the
   columns. Running headers and standalone page numbers are filtered out so they don't
   pollute the embeddings.

Pages with no usable text are skipped entirely.

In [ ]:
import pymupdf
import os


def extract_structured_page_data(pdf_path, max_pages=None):
    """Extract per-page text and figures from the manual.

    Returns a list of dicts: {"page": int, "text": str, "images": [paths]}
    """
    doc = pymupdf.open(pdf_path)
    extracted_docs = []

    pages_to_process = len(doc) if max_pages is None else min(max_pages, len(doc))
    print(f"Starting extraction for {pages_to_process} pages...")

    for page_num in range(pages_to_process):
        page = doc[page_num]

        # --- 1. Extract embedded figures for this page -------------------------
        images = page.get_images(full=True)
        page_image_paths = []
        for img_idx, img in enumerate(images):
            xref = img[0]
            base_img = doc.extract_image(xref)
            img_bytes = base_img["image"]
            img_ext = base_img["ext"]
            # Page number is baked into the filename so figures can be matched later
            img_name = f"page_{page_num + 1}_fig_{img_idx + 1}.{img_ext}"
            img_path = os.path.join(IMAGE_OUTPUT_DIR, img_name)

            with open(img_path, "wb") as f:
                f.write(img_bytes)
            page_image_paths.append(img_path)

        # --- 2. Extract text blocks, sorted by column layout -------------------
        blocks = page.get_text("blocks")
        blocks.sort(key=lambda b: (b[0], b[1]))  # sort by x0 then y0 to respect columns

        clean_blocks = []
        for b in blocks:
            text = b[4].strip()
            # Drop running headers and isolated page numbers: pure noise for retrieval
            if ("GETTING TO KNOW YOUR CAR" in text
                    or "STARTING AND DRIVING" in text
                    or text.isdigit()):
                continue
            clean_blocks.append(text)

        full_page_text = "\n\n".join(clean_blocks)

        # Keep only pages that actually carry content
        if full_page_text.strip():
            extracted_docs.append({
                "page": page_num + 1,
                "text": full_page_text,
                "images": page_image_paths,
            })

        if (page_num + 1) % 50 == 0:
            print(f"Processed {page_num + 1} pages...")

    return extracted_docs


# Run extraction across the whole document
parsed_pages = extract_structured_page_data(PDF_PATH)
print(f"\nProcessing complete. Valid pages extracted: {len(parsed_pages)}")

---
## 3. Rendering pages to PNG

Embedded figures alone are not enough: many pages carry diagrams drawn as vector graphics,
which `get_images()` does not return. Rendering every page at 150 DPI gives the API a
guaranteed visual fallback, so each answer can always show *something* from the manual.

This cell is self-contained (it re-mounts Drive and resolves the PDF path on its own) so
it can be run independently after a runtime restart.

In [ ]:
import os
import subprocess
from google.colab import drive

# 1. Make sure Drive is available (safe to re-run)
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. poppler-utils is the rendering backend required by pdf2image
print("Installing poppler-utils and pdf2image...")
subprocess.run(["apt-get", "update", "-y"], check=True)
subprocess.run(["apt-get", "install", "-y", "poppler-utils"], check=True)
subprocess.run(["pip", "install", "--upgrade", "-q", "pdf2image"], check=True)

from pdf2image import convert_from_path

# 3. Resolve the PDF path, falling back to any PDF present in the data folder
DATA_DIR = "/content/drive/MyDrive/RAG_Project/data"
PDF_PATH = os.path.join(DATA_DIR, "alfa_romeo_giulia_manual.pdf")

if not os.path.exists(PDF_PATH):
    print(f"\nFile not found at the default path. Searching inside {DATA_DIR}...")
    found_pdfs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
    if found_pdfs:
        PDF_PATH = os.path.join(DATA_DIR, found_pdfs[0])
        print(f"Found PDF: {PDF_PATH}")
    else:
        raise FileNotFoundError(f"No PDF file found inside {DATA_DIR}.")

PAGES_DIR = os.path.join(DATA_DIR, "manual_pages")
os.makedirs(PAGES_DIR, exist_ok=True)

# 4. Render every page at 150 DPI: readable for a human, light enough for Drive
print(f"\nConverting PDF pages to PNG from:\n{PDF_PATH}")
images = convert_from_path(PDF_PATH, dpi=150)

for i, image in enumerate(images):
    page_num = i + 1
    image_path = os.path.join(PAGES_DIR, f"page_{page_num}.png")
    image.save(image_path, "PNG")
    if page_num % 50 == 0 or page_num == len(images):
        print(f"Processed {page_num}/{len(images)} pages...")

print(f"\nDone. {len(images)} pages saved as PNG in: {PAGES_DIR}")

---
## 4. Chunking, embedding and vector storage

**Chunking.** 600 characters with 100 characters of overlap. Technical manual instructions
are short and self-contained, so small chunks keep retrieval precise; the overlap prevents
a procedure from being cut in half at a chunk boundary. The separator priority
(`paragraph → line → sentence → word`) makes splits land on natural boundaries.

**Metadata.** Every chunk carries its source page number and the paths of any figures on
that page. This is what makes citation and figure retrieval possible at query time —
without it, the answer could not point back at the manual.

**Embeddings.** `all-MiniLM-L6-v2`: 384 dimensions, fast enough to embed ~1,500 chunks on a
Colab T4 in under a minute, and strong enough for short technical passages.

**Storage.** ChromaDB with a persistent path on Drive, written in batches of 500 to keep
peak memory low.

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Persistent vector store location
VECTOR_DB_DIR = os.path.join(BASE_DIR, "vector_store")
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

# 2. Chunking strategy tuned for short, procedural technical text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""],
)

documents, metadatas, ids = [], [], []
chunk_id_counter = 0

print("Splitting text into chunks and attaching metadata...")

for page_data in parsed_pages:
    page_num = page_data["page"]
    page_text = page_data["text"]
    # Figure paths are stored as a comma-separated string: Chroma metadata must be scalar
    image_paths = ",".join(page_data["images"]) if page_data["images"] else "none"

    for chunk in text_splitter.split_text(page_text):
        documents.append(chunk)
        metadatas.append({"page": page_num, "images": image_paths})
        ids.append(f"doc_page_{page_num}_chunk_{chunk_id_counter}")
        chunk_id_counter += 1

print(f"Total chunks created: {len(documents)}")

# 3. Embedding function and collection
print("Generating embeddings and persisting to ChromaDB...")

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_or_create_collection(
    name="alfa_romeo_manual",
    embedding_function=sentence_transformer_ef,
)

# 4. Batched insertion keeps Colab memory usage flat
batch_size = 500
for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i:i + batch_size],
        metadatas=metadatas[i:i + batch_size],
        ids=ids[i:i + batch_size],
    )
    print(f"Inserted batch {i // batch_size + 1} / {(len(documents) - 1) // batch_size + 1}")

print("\nChromaDB vector store created and persisted on Google Drive.")

---
## 5. Enriching metadata with page-image paths

The vector store was built before the page renders existed, so each chunk now gets an
`image_path` pointing at its rendered page. This is what the API serves when a retrieved
page has no extracted figure.

One practical note: ChromaDB does heavy random I/O on SQLite, and doing that directly
against a mounted Google Drive causes lock errors and very slow updates. The fix is to copy
the store to local disk, update it there, then sync it back.

In [ ]:
import os
import shutil
import chromadb

# 1. Drive path (durable) vs local path (fast)
DRIVE_STORE_PATH = "/content/drive/MyDrive/RAG_Project/vector_store"
LOCAL_STORE_PATH = "/tmp/vector_store"

# 2. Copy the database to local storage to avoid Drive I/O locks
print("Copying vector_store to fast local temporary storage...")
if os.path.exists(LOCAL_STORE_PATH):
    shutil.rmtree(LOCAL_STORE_PATH)
shutil.copytree(DRIVE_STORE_PATH, LOCAL_STORE_PATH)

# 3. Open the local copy
chroma_client = chromadb.PersistentClient(path=LOCAL_STORE_PATH)
collections = chroma_client.list_collections()
print("Collections found:", [c.name for c in collections])

collection_name = collections[0].name if collections else "alfa_romeo_manual"
collection = chroma_client.get_collection(name=collection_name)

all_docs = collection.get()
ids = all_docs["ids"]
metadatas = all_docs["metadatas"]

# 4. Attach the rendered-page path to every chunk
if ids and metadatas:
    updated_metadatas = []
    for meta in metadatas:
        page_num = meta.get("page", 1)
        meta["image_path"] = f"manual_pages/page_{page_num}.png"
        updated_metadatas.append(meta)

    collection.update(ids=ids, metadatas=updated_metadatas)
    print(f"Updated metadata for {len(ids)} chunks in the local database.")

    # 5. Sync the updated database back to Drive
    print("Syncing updated database back to Google Drive...")
    shutil.rmtree(DRIVE_STORE_PATH)
    shutil.copytree(LOCAL_STORE_PATH, DRIVE_STORE_PATH)
    print("Database successfully updated on Google Drive.")
else:
    print("No documents found to update.")

---
## 6. Local LLM runtime (Ollama)

All generation runs locally. Ollama is installed, started as a background daemon, and a
lightweight model is pulled for the first prototype. `zstd` is installed first because the
Ollama installer needs it to unpack its payload on Colab images.

In [ ]:
import subprocess
import time

# 1. zstd is required by the Ollama installer on Colab
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server as a background process
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give the daemon a moment to bind its port

# 3. Pull a lightweight model for the first pipeline prototype
!ollama pull llama3.2

!pip install -q ollama

---
## 7. RAG pipeline prototype

A minimal in-notebook implementation used to validate retrieval quality before wrapping
anything in an API. Two deliberate choices in the prompt:

- Answers must come **only** from the retrieved context, and the model is told to say so
  explicitly when the manual does not cover the question. Hallucinated torque values or
  tyre pressures would make the system worse than useless.
- Page numbers are injected into the context as `[Source n | Page p]` labels so citations
  come from the retrieval layer rather than from the model's imagination.

In [ ]:
import os
import ollama
import chromadb
import pandas as pd

# Connect to the persistent store built in Section 4
VECTOR_DB_DIR = os.path.join(BASE_DIR, "vector_store")
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_collection(name="alfa_romeo_manual")


def query_rag_pipeline(question, n_results=3):
    """Retrieve the top-k chunks and generate a grounded, cited answer."""
    results = collection.query(query_texts=[question], n_results=n_results)

    retrieved_chunks = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]

    # Build a context string where each block is explicitly tagged with its page
    context_blocks, sources = [], []
    for idx, (chunk, meta) in enumerate(zip(retrieved_chunks, retrieved_metadata)):
        page_num = meta.get("page", "Unknown")
        sources.append(f"Page {page_num}")
        context_blocks.append(f"[Source {idx + 1} | Page {page_num}]:\n{chunk}")

    context_str = "\n\n".join(context_blocks)

    # Grounding instructions: answer from context only, cite pages, admit gaps
    prompt = f"""You are a helpful automotive technical assistant. Answer the question strictly using the provided context below.
If the context does not contain enough information, state clearly that the information is not available in the manual.
Always cite the source page number when providing facts.

Context:
{context_str}

Question: {question}

Answer:"""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response["message"]["content"]
    return answer, list(set(sources)), context_str


print("RAG pipeline initialised.")

### 7.1 Evaluation on a fixed query set

Ten questions chosen to span the different retrieval patterns the manual demands: exact
specification lookups (tyre pressure, oil grade), procedural instructions (activating a
driving mode), figure references, and conceptual explanations (what ESC and ASR do).
Failures are captured rather than raised so a single bad query never aborts the run, and
results are written to CSV for the report.

In [ ]:
import pandas as pd

# Fixed evaluation set covering specs, procedures, figures and concepts
test_queries = [
    "How do I activate the Advanced Efficiency mode?",
    "What is the recommended tire pressure for normal load?",
    "What kind of engine oil is recommended for Alfa Romeo Giulia?",
    "What does figure 164 display on the performance screen?",
    "How to turn off the Normal driving mode?",
    "What happens when the Electronic Q2 system is deactivated?",
    "How do I check the engine coolant level?",
    "What is the purpose of the ESC and ASR systems?",
    "How to operate the dynamic selector?",
    "What features are included in the Alfa Active Suspension (AAS)?",
]

evaluation_results = []
print("Running evaluation on 10 test queries...\n")

for idx, query in enumerate(test_queries, 1):
    print(f"[{idx}/{len(test_queries)}] {query}")
    try:
        answer, sources, context = query_rag_pipeline(query, n_results=3)
        evaluation_results.append({
            "Query ID": idx,
            "Question": query,
            "Answer": answer,
            "Sources": ", ".join(sources),
            "Status": "Success",
        })
    except Exception as e:
        # Log the failure and keep going: one bad query should not kill the run
        evaluation_results.append({
            "Query ID": idx,
            "Question": query,
            "Answer": f"Error: {e}",
            "Sources": "N/A",
            "Status": "Failed",
        })

eval_df = pd.DataFrame(evaluation_results)

# Persist results for the project report
EVAL_SAVE_PATH = os.path.join(BASE_DIR, "evaluation_results.csv")
eval_df.to_csv(EVAL_SAVE_PATH, index=False)
print("\nEvaluation complete. Results saved to:", EVAL_SAVE_PATH)

print("\n--- Summary of generated answers ---")
for _, row in eval_df.iterrows():
    print(f"\nQ{row['Query ID']}: {row['Question']}")
    print(f"Sources: {row['Sources']}")
    print(f"Answer : {row['Answer'][:200]}...")

---
## 8. FastAPI backend

The production service. Compared with the notebook prototype it adds:

- **A stronger reasoning model.** `qwen2.5:14b` replaces `llama3.2` for text queries; the
  larger model handles multi-step technical explanations noticeably better while still
  fitting on a T4.
- **Automatic vision routing.** If the user uploads an image, or the question mentions a
  figure or diagram, the retrieved figure is base64-encoded and routed to `llava:13b`
  instead. Text questions never pay the cost of the vision model.
- **Follow-up handling.** When a request contains several question marks, only the last
  question is embedded — otherwise a chat-style follow-up retrieves against the whole
  conversation history and returns noise.
- **Static mounts.** `/pages` and `/extracted` serve page renders and figures directly, so
  the client can render citations as images.
- **Low temperature (0.1).** Technical answers should be reproducible, not creative.

In [ ]:
# Pull the reasoning and vision models used by the API
import subprocess

print("Downloading qwen2.5:14b (text reasoning)...")
subprocess.run(["ollama", "pull", "qwen2.5:14b"])

print("Downloading llava:13b (technical figure analysis)...")
subprocess.run(["ollama", "pull", "llava:13b"])

!pip install -q fastapi uvicorn pyngrok pydantic

### 8.1 Writing `backend/app/main.py`

The application is written to Drive as a standalone file so it can be launched by
`uvicorn` and version-controlled independently of the notebook.

In [ ]:
import os
import time
import subprocess

BASE_DIR = "/content/drive/MyDrive/RAG_Project"
APP_DIR = os.path.join(BASE_DIR, "backend", "app")
os.makedirs(APP_DIR, exist_ok=True)

main_py_code = """import os
import base64
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
import chromadb
from sentence_transformers import SentenceTransformer
import ollama

app = FastAPI(title="Alfa Romeo Multimodal RAG")

# Open CORS: the client is served from a different origin (ngrok tunnel)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- Static image mounts ---------------------------------------------------
# Rendered full pages, used as the visual fallback for any cited page
PAGES_DIR = "/content/drive/MyDrive/RAG_Project/manual_pages"
if not os.path.exists(PAGES_DIR):
    PAGES_DIR = "/content/drive/MyDrive/RAG_Project/data/manual_pages"
app.mount("/pages", StaticFiles(directory=PAGES_DIR), name="pages")

# Figures extracted from the PDF, preferred over full-page renders when available
EXTRACTED_IMG_DIR = "/content/drive/MyDrive/RAG_Project/data/extracted_images"
if os.path.exists(EXTRACTED_IMG_DIR):
    app.mount("/extracted", StaticFiles(directory=EXTRACTED_IMG_DIR), name="extracted")

# --- Retrieval layer -------------------------------------------------------
VECTOR_STORE_PATH = "/content/drive/MyDrive/RAG_Project/vector_store"
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
collection = chroma_client.get_collection(name="alfa_romeo_manual")

# Must match the model used at indexing time
embedder = SentenceTransformer("all-MiniLM-L6-v2")


class QueryRequest(BaseModel):
    question: str
    n_results: int = 5
    image_b64: str = None


@app.get("/health")
def health_check():
    return {"status": "healthy"}


@app.post("/query")
def query_rag(request: QueryRequest):
    try:
        # Follow-up handling: embed only the latest question, not the whole thread
        raw_question = request.question.strip()
        if "?" in raw_question[:-1]:
            parts = [p.strip() for p in raw_question.split("?") if p.strip()]
            active_question = parts[-1] + "?" if parts else raw_question
        else:
            active_question = raw_question

        # --- Vector search -------------------------------------------------
        query_embedding = embedder.encode(active_question).tolist()
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=request.n_results,
        )

        retrieved_docs = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []

        sources = []
        figures = []
        target_image_path = None  # first figure found, candidate for visual analysis

        for meta in metadatas:
            page_num = meta.get('page', 'Unknown')
            sources.append("Page " + str(page_num))

            # Prefer a real extracted figure for this page
            figure_found = False
            if os.path.exists(EXTRACTED_IMG_DIR):
                extracted_files = os.listdir(EXTRACTED_IMG_DIR)
                matching_figs = [f for f in extracted_files
                                 if f.startswith("page_" + str(page_num) + "_")]
                if matching_figs:
                    fig_filename = matching_figs[0]
                    if not target_image_path:
                        target_image_path = os.path.join(EXTRACTED_IMG_DIR, fig_filename)
                    figures.append({
                        "src": "/extracted/" + fig_filename,
                        "caption": "Alfa Romeo Technical Figure - Page " + str(page_num),
                    })
                    figure_found = True

            # Otherwise fall back to the rendered full page
            if not figure_found:
                image_path = meta.get('image_path')
                if image_path:
                    filename = image_path.split('/')[-1]
                    if not target_image_path:
                        target_image_path = image_path
                    figures.append({
                        "src": "/pages/" + filename,
                        "caption": "Alfa Romeo Manual - Page " + str(page_num),
                    })

        # Deduplicate figures while preserving retrieval ranking
        unique_figures = []
        seen_srcs = set()
        for fig in figures:
            if fig["src"] not in seen_srcs:
                seen_srcs.add(fig["src"])
                unique_figures.append(fig)

        context_text = "\\n\\n".join(retrieved_docs)

        # Grounding contract: manual only, structured output, explicit refusal string
        system_instruction = (
            "You are an expert Automotive Chief Engineer and Technical Illustrator for "
            "the Alfa Romeo Giulia. Analyze the provided manual context and technical "
            "figures thoroughly. If the user asks to explain a diagram, component layout "
            "or figure, break down its visual parts logically, explaining what each label, "
            "component or section represents based on the technical manual. Use clear "
            "bullet points. If information is missing, output exactly: "
            "'I cannot find this information in the manual.'"
        )

        prompt = (
            system_instruction
            + "\\n\\nContext from manual:\\n" + context_text
            + "\\n\\nUser Question: " + active_question
            + "\\nEngineering Figure & Text Analysis:"
        )

        # --- Model routing --------------------------------------------------
        # Vision model is used only when the question is actually visual,
        # so plain text queries keep the faster reasoning model.
        visual_keywords = ["figure", "diagram", "show", "image", "explain this",
                           "illustration", "layout"]
        is_requesting_visual = any(k in active_question.lower() for k in visual_keywords)

        model_name = "qwen2.5:14b"
        image_payload_b64 = request.image_b64

        if request.image_b64:
            # User supplied an image: always go multimodal
            model_name = "llava:13b"
        elif target_image_path and os.path.exists(target_image_path) and is_requesting_visual:
            # Question is visual and a figure was retrieved: encode and analyse it
            with open(target_image_path, "rb") as img_file:
                image_payload_b64 = base64.b64encode(img_file.read()).decode("utf-8")
            model_name = "llava:13b"

        messages = [{'role': 'user', 'content': prompt}]
        if image_payload_b64:
            messages[0]['images'] = [image_payload_b64]

        response = ollama.chat(
            model=model_name,
            messages=messages,
            options={"temperature": 0.1, "num_predict": 600},  # low temp = reproducible specs
        )

        return {
            "answer": response['message']['content'],
            "sources": list(set(sources)),
            "figures": unique_figures[:2],
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""

with open(os.path.join(APP_DIR, "main.py"), "w") as f:
    f.write(main_py_code)

print("Backend written to:", os.path.join(APP_DIR, "main.py"))

### 8.2 Launching the server and exposing it

`uvicorn` runs in the background on port 8000 and ngrok publishes it.

**The ngrok token is read from the environment — never hard-coded.** In Colab, add it via
the key icon in the left sidebar (*Secrets* → name it `NGROK_AUTHTOKEN`), or export it
before running. If neither is available the cell prompts for it and the value is not
stored in the notebook.

In [ ]:
import os
import time
import subprocess
from getpass import getpass
from pyngrok import ngrok

APP_DIR = "/content/drive/MyDrive/RAG_Project/backend/app"

# 1. Free port 8000 in case a previous server is still running
os.system("fuser -k 8000/tcp")
time.sleep(2)

# 2. Start the FastAPI server in the background
print("Starting FastAPI backend...")
backend_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# 3. Wait for ChromaDB and the embedding model to load, then check for an early crash
time.sleep(15)
if backend_process.poll() is not None:
    print("Server failed to start. Log below:")
    print(backend_process.stdout.read())
else:
    print("Server process is alive. Testing /health endpoint:")
    !curl -s http://127.0.0.1:8000/health

# 4. Resolve the ngrok token from the environment (Colab secret, env var, or prompt)
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN")

if not NGROK_AUTHTOKEN:
    try:
        from google.colab import userdata
        NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
    except Exception:
        NGROK_AUTHTOKEN = None

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = getpass("Enter your ngrok auth token (input hidden): ")

# 5. Open the public tunnel
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()  # close any tunnel left over from a previous run

public_url = ngrok.connect(8000).public_url
print(f"\nPublic API URL: {public_url}")
print(f"Health check  : {public_url}/health")

---
## 9. Smoke test

Confirms the models are registered with Ollama and that an end-to-end query returns a
grounded answer through the API.

In [ ]:
import requests

# 1. Confirm the expected models are available locally
!ollama list

# 2. End-to-end request against the running API
print("\nTesting /query endpoint:")
try:
    res = requests.post(
        "http://127.0.0.1:8000/query",
        json={"question": "What is the recommended tire pressure?", "n_results": 3},
        timeout=120,
    )
    print("Status code:", res.status_code)
    payload = res.json()
    print("Sources    :", payload.get("sources"))
    print("Figures    :", [f["src"] for f in payload.get("figures", [])])
    print("Answer     :", payload.get("answer", "")[:400], "...")
except Exception as e:
    print("Error:", e)

---
## 10. Design notes and results

### Why these choices

**Page-level parsing instead of whole-document parsing.** Retrieval is only useful here if
the answer can be traced back to a page, so the page number is treated as a first-class
key from the very first step — it flows from the parser into chunk metadata, into the API
response, and finally into the figure filename convention.

**Column-aware block sorting.** The manual is laid out in two columns. Default extraction
order interleaves them and produces sentences that read like two unrelated instructions
spliced together, which quietly corrupts the embeddings. Sorting blocks by `(x0, y0)`
fixes this at the source.

**Small chunks with overlap.** Manual instructions are dense and short. Large chunks dilute
the embedding with unrelated procedures; the 100-character overlap avoids severing a
procedure at a boundary.

**Two image strategies, not one.** Extracted raster figures are precise but incomplete
(vector diagrams are invisible to `get_images()`). Full-page renders always exist but are
coarse. The API prefers the figure and falls back to the page, so every answer can show
its source.

**Conditional vision routing.** Running a vision model on every request is wasteful — most
questions are textual. Routing to `llava:13b` only when an image is uploaded or the
question is phrased visually keeps median latency down.

**Refusal over hallucination.** Wrong tyre pressures or oil grades are worse than no
answer, so the prompt mandates a fixed refusal string when the context is insufficient.

### Results

| Metric | Value |
|---|---|
| Source pages processed | 360 |
| Pages with extractable content | ~350 |
| Chunks indexed | 1,561 |
| Embedding model | `all-MiniLM-L6-v2` (384-dim) |
| Retrieval depth | top-3 (notebook), top-5 (API) |
| Evaluation queries | 10 / 10 answered with page citations |
| Generation models | `qwen2.5:14b` (text), `llava:13b` (vision) |

Per-query outputs are written to `evaluation_results.csv` by Section 7.1.

### Known limitations

- **Dense retrieval only.** No BM25 or hybrid search, so exact-token lookups such as a part
  number can be missed. Hybrid retrieval with reciprocal-rank fusion is the obvious next
  step.
- **No reranker.** A cross-encoder over the top-20 candidates would improve precision at
  top-3.
- **Figure matching is filename-based.** A page can hold several figures and only the first
  is attached; figure captions are not parsed, so "figure 164" is matched by text, not by
  an actual figure index.
- **No automatic answer scoring.** Evaluation is qualitative. Faithfulness and
  context-relevance metrics (e.g. RAGAS) would make regressions measurable.
- **Colab-bound paths.** Drive paths are hard-coded; moving to a container would mean
  replacing them with environment variables.

---
## Appendix A — Restart recovery

Colab drops the runtime after inactivity, which kills Ollama and the API but leaves the
vector store on Drive intact. This single cell rebuilds the runtime from scratch: mount,
install, start Ollama, relaunch the backend. Run Section 8.2 afterwards to reopen the
tunnel.

In [ ]:
import os
import subprocess
import time
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/RAG_Project'
APP_DIR = os.path.join(BASE_DIR, "backend", "app")

# 2. Reinstall the runtime dependencies (the container is fresh, Drive is not)
print("Installing dependencies...")
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama chromadb sentence-transformers langchain-text-splitters fastapi uvicorn pyngrok pydantic

# 3. Restart Ollama and pull the serving models
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull qwen2.5:14b
!ollama pull llava:13b

# 4. Relaunch the backend
print("Starting FastAPI backend...")
!fuser -k 8000/tcp || true
time.sleep(2)

backend_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(15)

print("Testing health endpoint:")
!curl -s http://127.0.0.1:8000/health

---
## Appendix B — Earlier backend iteration (text-only)

Kept for reference. This was the first API version: `qwen2.5:14b` for every request, with
the vision model used only when the client explicitly uploaded an image. Section 8
supersedes it by adding automatic figure routing. Do not run this cell unless you want to
overwrite `main.py` with the older behaviour.

```python
# Model selection in v1 — reactive only:
model_name = "llama3.2-vision" if request.image_b64 else "qwen2.5:14b"

# Model selection in v2 (Section 8) — reactive plus intent-aware:
if request.image_b64:
    model_name = "llava:13b"
elif target_image_path and is_requesting_visual:
    image_payload_b64 = base64.b64encode(open(target_image_path, "rb").read()).decode()
    model_name = "llava:13b"
else:
    model_name = "qwen2.5:14b"
```